# User Experience para ciência de dados
Previsão de atrasos em pedidos.

In [113]:
import pandas as pd

# Força limpeza
import importlib
import data_preparation_final
importlib.reload(data_preparation_final)

# Importar lógica de preparação de dados existente
from data_preparation_final import load_and_clean_data

In [114]:
def prepare_data_for_lgbm(df):
    """
    Limpeza e filtragem sugerida para o modelo LightGBM.
    """
    print("\n--- Preparação de Dados para LightGBM ---")
    inicial = len(df)
    
    # 1. Limpeza Crítica: dropna em campos fundamentais
    cols_limpeza = ['dt_despacho_pedido', 'dt_entrega_pedido', 'dt_pagamento_pedido', 'qtd_dias_tat']
    
    # --- OPÇÃO DE IMPUTAÇÃO (COMENTADA) ---
    # Se decidirmos não dropar os nulos de dt_pagamento_pedido (6.4%):
    # df['dt_pagamento_pedido'] = df['dt_pagamento_pedido'].fillna(method='ffill') # ou uma data fixa 'Pendente'
    # df['dias_aprovacao'] = df['dias_aprovacao'].fillna(-1) # Categoria para 'Pendente'
    # --------------------------------------
    
    df = df.dropna(subset=cols_limpeza).copy()
    
    posterior = len(df)
    print(f"Registros antes: {inicial}")
    print(f"Registros após limpeza (dropna): {posterior}")
    print(f"Perda de dados: {1 - (posterior/inicial):.2%}")

    # 2. Tratamento de Outliers (Percentil 99 em qtd_dias_tat)
    limite_99 = df['qtd_dias_tat'].quantile(0.99)
    df = df[df['qtd_dias_tat'] <= limite_99].copy()
    print(f"Outliers removidos (TAT > {limite_99:.1f} dias): {posterior - len(df)}")

    return df

## Limpeza e tratamento de dados

In [123]:
# 1. Carregar e Limpar
input_file = "pedidos_logistica.parquet"
df = load_and_clean_data(input_file, drop_ids=False)

if df is not None:
    df = prepare_data_for_lgbm(df)

Carregando dados de pedidos_logistica.parquet...
Colunas após renomeação:
id, cod_pedido, uf
grp_transportadora, dt_despacho_pedido, dt_entrega_pedido
dt_previsao_entrega_cliente, dt_criacao, dt_pagamento_pedido
flg_existem_ocorrencias, tp_praca, des_unidade_negocio
des_cd_origem, qtd_dias_tat, tp_performance_entrega
cidade_destinatario
Removendo registros inconsistentes (entrega antecede despacho): 7
Realizando engenharia de features...
Processamento concluído. Formato final: (490210, 21)
Salvando dataset limpo em pedidos_logistica_limpo.parquet...

--- Preparação de Dados para LightGBM ---
Registros antes: 490210
Registros após limpeza (dropna): 458687
Perda de dados: 6.43%
Outliers removidos (TAT > 14.0 dias): 3596


In [131]:
# Manter para ter coerencia com 'trabalho' - Pedro
df = df.drop(
    columns=[
        # 'row_id',
        # 'hr_despacho_pedido',
        'dt_entrega_pedido',
        # 'hr_entrega_pedido',
        'flg_existem_ocorrencias'
    ],
    errors='ignore'
 )

df.head(10)

,id,cod_pedido,uf,grp_transportadora,dt_despacho_pedido,dt_previsao_entrega_cliente,dt_criacao,dt_pagamento_pedido,tp_praca,des_unidade_negocio,des_cd_origem,qtd_dias_tat,tp_performance_entrega,cidade_destinatario,dias_gastos_cd,dias_transito,dias_atraso_real,dias_ciclo,dias_restantes_prazo
296320,364385,121676962-1,MT,Transportadora 2,2023-07-03 13:19:31,2023-07-25,2023-07-01,2023-07-01,Interior,Mono,PR-Campina G. Sul,8.0,1,NOVA NAZARE,2.555220,9.235289,-12.209491,11.790509,24.0
388636,456754,121668868-1,BA,Transportadora 1,2023-07-03 14:03:38,2023-07-20,2023-07-01,2023-07-01,Interior,Mono,PR-Campina G. Sul,5.0,1,CARDEAL DA SILVA,2.585856,5.100093,-11.314051,7.685949,19.0
209761,277532,121674465-1,MG,Transportadora 1,2023-07-03 13:38:43,2023-07-10,2023-07-01,2023-07-01,Interior,Mono,PR-Campina G. Sul,3.0,1,SANTA RITA DO SAPUCAI,2.568553,1.868333,-4.563113,4.436887,9.0
233401,301266,121676089-1,DF,Transportadora 4,2023-07-03 15:00:40,2023-07-06,2023-07-01,2023-07-01,Capital,Mono,PR-Campina G. Sul,3.0,1,BRASILIA,2.625463,1.889097,-0.485440,4.514560,5.0
57140,124402,121662678-1,BA,Transportadora 3,2023-07-01 06:42:00,2023-07-11,2023-07-01,2023-06-30,Capital,Multi,PR-Campina G. Sul,4.0,1,CAMACARI,1.279167,5.136100,-4.584734,5.415266,11.0
105125,172580,121671479-1,MG,Transportadora 1,2023-07-03 13:46:30,2023-07-11,2023-07-01,2023-07-01,Interior,Mono,PR-Campina G. Sul,5.0,1,SAO SEBASTIAO DO PONTAL,2.573958,3.867940,-3.558102,6.441898,10.0
276572,344566,121674252-1,SP,Transportadora 1,2023-07-03 09:43:56,2023-07-07,2023-07-01,2023-07-01,Interior,Mono,SP-Registro,4.0,1,CHARQUEADA,2.405509,3.286829,-0.307662,5.692338,6.0
16717,83843,121656740-2,SP,Transportadora 5,2023-07-01 12:35:30,2023-07-06,2023-07-01,2023-06-30,Capital,Multi,PR-Campina G. Sul,1.0,1,SAO PAULO,1.524653,1.978113,-2.497234,2.502766,6.0
435507,387,121681179-1,MG,Transportadora 1,2023-07-03 14:02:10,2023-07-06,2023-07-01,2023-07-01,Reg. Metropolitana,Mono,PR-Campina G. Sul,4.0,1,RIBEIRAO DAS NEVES,2.584838,3.006620,0.591458,5.591458,5.0
375017,443082,121667959-1,MG,Transportadora 1,2023-07-03 09:43:42,2023-07-11,2023-07-01,2023-07-01,Interior,Mono,SP-Registro,4.0,1,PRADOS,2.405347,3.216528,-4.378125,5.621875,10.0


## Profiling de dados

In [132]:
# profile = ProfileReport(df, title="Profiling Report")
# html = profile.to_html()
# output_file = 'report.html'
# with open(output_file, 'w') as f:
#     f.write(html)

In [ ]:
from sklearn.model_selection import train_test_split
import lightgbm as lgb

# TODO adaptar modelo abaixo com essa funcao
def train_lgbm(df):
    """
    Treinamento do modelo LightGBM usando a API nativa e recursos de classificação.
    """
    # 1. Divisão Temporal 70/30
    df = df.sort_values('dt_criacao')
    split_idx = int(len(df) * 0.7)
    train_test_data = df.iloc[:split_idx].copy()
    holdout_data = df.iloc[split_idx:].copy()    
    
    # 2. Definição de Features e Alvo
    # Categóricas (Nativas do LightGBM)
    cat_features = [
        'cidade_destinatario', 
        'uf', 
        'grp_transportadora', 
        'dt_pagamento_pedido',
        'dt_previsao_entrega_cliente',
        'dt_criacao',
        'tp_praca', 
        'des_unidade_negocio', 
        'des_cd_origem'
    ]
    # Numéricas
    # num_features = ['dias_aprovacao']
    num_features = []
    
    target = 'tp_performance_entrega'
    
    # Converter categóricas para o tipo 'category' do Pandas (Exigência do LightGBM)
    for col in cat_features:
        train_test_data[col] = train_test_data[col].astype('category')
        holdout_data[col] = holdout_data[col].astype('category')
    
    features = cat_features + num_features

    X = train_test_data[features]
    y = train_test_data[target]
    
    # Split para Treino e Validação (dentro dos 70%)
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # Cálculo do scale_pos_weight para desbalanceamento
    # Justificativa: LightGBM lida melhor com classes minoritárias se aumentarmos o peso dos positivos (atrasos).
    pos_count = y_train.sum()
    neg_count = len(y_train) - pos_count
    spw = neg_count / pos_count
    
    print(f"\nConfigurando scale_pos_weight: {spw:.2f} (Classe 'Atrasado' é minoritária)")

    # 4. Configuração do Modelo
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'scale_pos_weight': spw,
        'learning_rate': 0.05,
        'num_leaves': 128,
        'max_depth': -1,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'seed': 42
    }

    print("Iniciando treinamento com LightGBM...")
    
    train_set = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_features)
    val_set = lgb.Dataset(X_val, label=y_val, reference=train_set, categorical_feature=cat_features)
    
    model = lgb.train(
        params,
        train_set,
        valid_sets=[train_set, val_set],
        num_boost_round=1000,
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=50)
        ]
    )

    return model, holdout_data, features

In [138]:

import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ---------------------------------------------------------
# Step 3: Split Features (X) and Target (y)
# ---------------------------------------------------------
selected_features = [
    'cidade_destinatario',
    'uf',
    'grp_transportadora',
    'dt_previsao_entrega_cliente',
    'dt_criacao',
    'dt_pagamento_pedido',
    'tp_praca',
    'des_unidade_negocio',
    'des_cd_origem',
]

# 1. Divisão Temporal 70/30
df = df.sort_values('dt_criacao')
split_idx = int(len(df) * 0.7)
# df de treinamento e avaliação do modelo
train_test_data = df.iloc[:split_idx].copy()
# df para simulação (dados não conhecidos pelo modelo)
holdout_data = df.iloc[split_idx:].copy()

available_features = [col for col in selected_features if col in train_test_data.columns]
missing_features = [col for col in selected_features if col not in train_test_data.columns]

if missing_features:
    print('Missing columns (ignored):', missing_features)

X = train_test_data[available_features].copy()
y = train_test_data['tp_performance_entrega']

# Remove rows with missing target
valid_mask = y.notna()
X = X.loc[valid_mask].copy()
y = y.loc[valid_mask].astype('int32').copy()

# Convert requested date columns to numeric representation (ordinal days)
# date_cols = ['dt_previsao_entrega_cliente', 'dt_criacao', 'dt_pagamento_pedido']
# for col in [c for c in date_cols if c in X.columns]:
#     X[col] = pd.to_datetime(X[col], errors='coerce')
#     X[col] = X[col].map(lambda x: x.toordinal() if pd.notna(x) else np.nan).astype('float32')

# !!! Tratamento de Datas (Ponto de Atenção)
# Você está convertendo datas para toordinal(). Para logística, o número ordinal puro (ex: 738900) 
# é difícil para o modelo entender sazonalidade. O ganho de informação (gain) nessas colunas costuma ser baixo, 
# o que contribui para o erro de "no further splits".
# Dica para o Semáforo: Em vez de apenas o número ordinal, crie colunas de diferença (lead time):
# Exemplo de Feature Engineering útil para logística:
# X['dias_previsao_criacao'] = (X['dt_previsao_entrega_cliente'] - X['dt_criacao'])
# X['dias_pagamento_criacao'] = (X['dt_pagamento_pedido'] - X['dt_criacao'])

# Encode requested categorical columns as numeric codes
categorical_cols = [
    'cidade_destinatario',
    'uf',
    'grp_transportadora',
    'tp_praca',
    'des_unidade_negocio',
    'des_cd_origem',
]
# for col in [c for c in categorical_cols if c in X.columns]:
#     X[col] = X[col].astype('category').cat.codes.replace(-1, np.nan).astype('float32')
# !!! Categorização Manual vs. Nativa
# Você está usando .cat.codes. O LightGBM tem um suporte nativo excelente para categorias que performa muito melhor do que converter para números ordinais.
# Como melhorar:
# Mantenha as colunas como o tipo category do pandas.
# Não use .cat.codes.
# Passe a lista de nomes das colunas para o modelo.
# No seu loop de categorias, pare aqui:
for col in categorical_cols:
    X[col] = X[col].astype('category')

# LightGBM does not accept object/datetime columns directly
unsupported_cols = X.select_dtypes(include=['object', 'datetime64[ns]', 'datetimetz']).columns
if len(unsupported_cols) > 0:
    print('Dropping unsupported columns:', list(unsupported_cols))
X = X.drop(columns=unsupported_cols, errors='ignore')

print('Training columns:', list(X.columns))

# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------------
# Step 4: Initialize and Train the Model
# ---------------------------------------------------------
# model = lgb.LGBMClassifier(
#     n_estimators=100,
#     learning_rate=0.1,
#     max_depth=5,
#     random_state=42,
#     is_unbalance=True
# )
# Sobre a causa das mensagens " LightGBM Warning No further splits with positive gain, best gain: -inf"
# Com mais de 400 mil registros, o problema certamente não é falta de dados (o min_child_samples=20 padrão seria atendido facilmente). 
# O aviso aparece porque você está limitando muito o modelo com max_depth=5 em um cenário de logística que parece ser complexo.
# Com mais de 400k linhas, uma profundidade de 5 níveis (máximo de 32 folhas) é muito pouco para capturar as nuances de logística 
# (como variações por cidade ou transportadora). O LightGBM tenta criar divisões, mas como ele já atingiu o limite de profundidade ou 
# as combinações restantes não batem com o is_unbalance=True, ele desiste e gera o aviso.

# Sugestão: Deixe o modelo crescer mais e controle pelo número de folhas, que é mais eficiente no LightGBM.
model = lgb.LGBMClassifier(
    n_estimators=200,      # Aumente um pouco já que tem muitos dados
    learning_rate=0.05,    # Reduza a taxa para aprender com mais calma
    num_leaves=63,         # Aumente a complexidade (2^max_depth - 1)
    max_depth=-1,          # Deixe o crescimento livre (controlado por num_leaves)
    random_state=42,
    is_unbalance=True,
    verbose=-1             # Silencie o aviso agora que ajustamos a estrutura
)

# Fit model
model.fit(X_train, y_train)

# ---------------------------------------------------------
# Step 5: Make Predictions (The "Risk Score")
# ---------------------------------------------------------
predictions_binary = model.predict(X_test)
predictions_proba = model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# Step 6: Evaluate Model Quality
# ---------------------------------------------------------
accuracy = accuracy_score(y_test, predictions_binary)
precision = precision_score(y_test, predictions_binary, zero_division=0)
recall = recall_score(y_test, predictions_binary, zero_division=0)
f1 = f1_score(y_test, predictions_binary, zero_division=0)
roc_auc = roc_auc_score(y_test, predictions_proba)
cm = confusion_matrix(y_test, predictions_binary)

print('=== Model Metrics ===')
print(f'Accuracy : {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1-score : {f1:.4f}')
print(f'ROC-AUC  : {roc_auc:.4f}')
print('\nConfusion Matrix:')
print(cm)
print('\nClassification Report:')
print(classification_report(y_test, predictions_binary, zero_division=0))

Dropping unsupported columns: ['dt_previsao_entrega_cliente', 'dt_criacao', 'dt_pagamento_pedido']
Training columns: ['cidade_destinatario', 'uf', 'grp_transportadora', 'tp_praca', 'des_unidade_negocio', 'des_cd_origem']
=== Model Metrics ===
Accuracy : 0.7242
Precision: 0.9833
Recall   : 0.7270
F1-score : 0.8360
ROC-AUC  : 0.7277

Confusion Matrix:
[[ 1364   759]
 [16812 44778]]

Classification Report:
              precision    recall  f1-score   support

           0       0.08      0.64      0.13      2123
           1       0.98      0.73      0.84     61590

    accuracy                           0.72     63713
   macro avg       0.53      0.68      0.49     63713
weighted avg       0.95      0.72      0.81     63713



In [139]:
# import matplotlib.pyplot as plt

# fig, ax = plt.subplots(figsize=(10, 6))
# lgb.plot_importance(model, ax=ax)
# plt.tight_layout()
# plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
# plt.close(fig)
# print('Saved: feature_importance.png')

In [140]:
# ---------------------------------------------------------
# Step 7: View the Results
# ---------------------------------------------------------
results_df = X_test.copy()
results_df['probabilidade_atraso'] = predictions_proba
results_df['risco_semaforo'] = pd.cut(
    results_df['probabilidade_atraso'],
    bins=[-0.1, 0.3, 0.7, 1.1],
    labels=['🟢 Verde', '🟡 Amarelo', '🔴 Vermelho']
)
print(results_df[['probabilidade_atraso', 'risco_semaforo']].head(20))

        probabilidade_atraso risco_semaforo
175920              0.475210      🟡 Amarelo
46887               0.745728     🔴 Vermelho
98681               0.170535        🟢 Verde
154178              0.452509      🟡 Amarelo
254993              0.654800      🟡 Amarelo
274410              0.981262     🔴 Vermelho
71996               0.986110     🔴 Vermelho
292686              0.976155     🔴 Vermelho
239858              0.464562      🟡 Amarelo
77209               0.326573      🟡 Amarelo
325701              0.801441     🔴 Vermelho
34930               0.995499     🔴 Vermelho
437855              0.955003     🔴 Vermelho
297679              0.995139     🔴 Vermelho
113941              0.979704     🔴 Vermelho
39442               0.267235        🟢 Verde
416884              0.860954     🔴 Vermelho
462117              0.207933        🟢 Verde
314139              0.315061      🟡 Amarelo
499547              0.530843      🟡 Amarelo


In [143]:
import joblib

# Build categorical mappings from training data
categorical_mappings = {}
for col in [c for c in categorical_cols if c in train_test_data.columns]:
    cats = pd.Series(train_test_data.loc[valid_mask, col].astype("string").dropna().unique()).sort_values().tolist()
    categorical_mappings[col] = {v: i for i, v in enumerate(cats)}

bundle = {
    "model": model,
    "selected_features": selected_features,
    "date_cols": ["dt_previsao_entrega_cliente", "dt_criacao", "dt_pagamento_pedido"],
    "categorical_cols": [c for c in categorical_cols if c in selected_features],
    "categorical_mappings": categorical_mappings,
    "threshold": 0.5
}

joblib.dump(bundle, "model_bundle.joblib")
print("Saved model_bundle.joblib")

Saved model_bundle.joblib


In [ ]:
# df[df['tp_performance_entrega'] == 0]

# Da parte do df(70%) utilizado no treino/teste
train_test_data[train_test_data['tp_performance_entrega'] == 0]
# len(train_test_data)

,id,cod_pedido,uf,grp_transportadora,dt_despacho_pedido,dt_previsao_entrega_cliente,dt_criacao,dt_pagamento_pedido,tp_praca,des_unidade_negocio,des_cd_origem,qtd_dias_tat,tp_performance_entrega,cidade_destinatario,dias_gastos_cd,dias_transito,dias_atraso_real,dias_ciclo,dias_restantes_prazo
412661,480856,121665744-1,PE,Transportadora 2,2023-07-03 07:51:39,2023-07-11,2023-07-01,2023-07-01,Capital,Mono,PR-Campina G. Sul,8.0,0,RECIFE,2.327535,9.543461,1.870995,11.870995,10.0
421495,489770,121662892-1,PE,Transportadora 2,2023-07-01 09:04:39,2023-07-13,2023-07-01,2023-06-30,Capital,Multi,PR-Campina G. Sul,10.0,0,JABOATAO DOS GUARARAPES,1.378229,13.269676,1.647905,13.647905,13.0
412098,480287,121672466-1,SP,Transportadora 1,2023-07-03 13:46:02,2023-07-06,2023-07-01,2023-07-01,Interior,Mono,PR-Campina G. Sul,5.0,0,AMERICANA,2.573634,4.078981,1.652616,6.652616,5.0
412279,480469,121681163-1,SP,Transportadora 1,2023-07-03 13:37:15,2023-07-07,2023-07-01,2023-07-01,Interior,Mono,PR-Campina G. Sul,6.0,0,JABOTICABAL,2.567535,7.257153,3.824688,9.824687,6.0
413927,482130,121682750-1,SP,Transportadora 1,2023-07-03 16:20:59,2023-07-05,2023-07-01,2023-07-01,Capital,Mono,SP-Registro,6.0,0,SAO PAULO,2.681238,6.795718,5.476956,9.476956,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420123,488386,129155069-1,RS,Transportadora 1,2024-01-03 12:16:06,2024-01-05,2023-12-31,2023-12-31,Capital,Multi,SP-Cajamar,5.0,0,PORTO ALEGRE,3.511181,4.000069,2.511250,7.511250,5.0
420208,488472,129153276-2,SP,Transportadora 3,2024-01-05 20:33:45,2024-01-08,2023-12-31,2023-12-31,Interior,Multi,SP-Cajamar,9.0,0,IBATE,5.856771,5.599016,3.455787,11.455787,8.0
419878,488140,129146283-1,RS,Transportadora 4,2024-01-03 09:02:20,2024-01-05,2023-12-31,2023-12-31,Capital,Mono,PR-Campina G. Sul,5.0,0,PORTO ALEGRE,3.376620,3.149722,1.526343,6.526343,5.0
433727,502131,129158916-2,SP,Transportadora 1,2024-01-04 19:26:22,2024-01-04,2023-12-31,2023-12-31,Capital,Multi,SP-Cajamar,5.0,0,SAO PAULO,4.809977,0.921238,1.731215,5.731215,4.0
